# Preparación de Estructura

In [1]:
!pip install -q -U transformers==4.46.2 accelerate==1.1.1 peft==0.13.2 bitsandbytes>=0.44.1 pillow==10.4.0 \
    chromadb sentence-transformers rank-bm25 pandas gradio

import os, shutil, json, gc, re
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageEnhance, ImageOps

from google.colab import drive
drive.mount("/content/drive")

REPO_DIR = "Isabela_Mendoza_Juan_Manuel_Lopera_Cesar_Montoya"
if not os.path.exists("notebooks/M4") and not os.path.exists(REPO_DIR):
    !git clone -b juanma https://github.com/CesarMontoyag1/Isabela_Mendoza_Juan_Manuel_Lopera_Cesar_Montoya.git
if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)

os.makedirs("final/corpus", exist_ok=True)
os.makedirs("final/eval", exist_ok=True)
os.makedirs("final/resultados", exist_ok=True)
os.makedirs("final/imagenes_test", exist_ok=True)

# Copiar corpus, eval_set y adaptador LoRA para que final/ sea 100% autocontenido
shutil.copy("notebooks/M3/corpus/corpus_look_es.csv", "final/corpus/")
shutil.copy("notebooks/M3/corpus/corpus_label_es.csv", "final/corpus/")
shutil.copy("notebooks/M4/eval/eval_set_m4.json", "final/eval/")
shutil.copytree("notebooks/M4/imagenes_test", "final/imagenes_test", dirs_exist_ok=True)

ADAPTER_DRIVE = "/content/drive/MyDrive/personal-shopper-ia/qwen25-1.5b-style-extractor-lora/adapter_final"
if os.path.exists(ADAPTER_DRIVE):
    shutil.copytree(ADAPTER_DRIVE, "final/adapter_final", dirs_exist_ok=True)

# Ajustar rutas de imagen en final/eval/eval_set_m4.json para que apunten a final/imagenes_test/
with open("final/eval/eval_set_m4.json", encoding="utf-8") as f:
    eval_m5 = json.load(f)
for item in eval_m5:
    if item.get("image_path"):
        item["image_path"] = item["image_path"].replace("notebooks/M4/", "final/")
with open("final/eval/eval_set_m4.json", "w", encoding="utf-8") as f:
    json.dump(eval_m5, f, ensure_ascii=False, indent=2)

print("Estructura autocontenida final/ preparada.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Mounted at /content/drive
Cloning into 'Isabela_Mendoza_Juan_Manuel_Lopera_Cesar_Montoya'...
remote: Enumerating objects: 368, done.
remote: Counting objects: 100% (368/368), done.
remote: Compressing objects: 100% (276/276), done.
remote: Total 368 (delta 158), reused 242 (delta 83), pack-reused 0 (from 0)
Receiving objects: 100% (368/368), 5.72 MiB | 17.23 MiB/s, done.
Resolving deltas: 100% (158/158), done.
Estructura autocontenida final/ preparada.


# Ejecutar Evaluación M5 sobre la Misma Vara que M4 (y Medir Invarianza)

In [2]:
import sys
sys.path.insert(0, os.path.abspath("final"))

from transformers import AutoModelForCausalLM, AutoTokenizer
from pipeline import PersonalShopperOptimizado, VALVULA, parse_json_seguro

# Reiniciar contador de VRAM para medir exclusivamente el sistema M5 optimizado
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# Evaluamos sin cache_hit artificial para medir latencia real en frío por caso
shopper_m5 = PersonalShopperOptimizado(usar_4bit=True, usar_cache=False)
print("Health Check M5:", shopper_m5.health_check())

with open("final/eval/eval_set_m4.json", encoding="utf-8") as f:
    EVAL_SET_M5 = json.load(f)

respuestas_m5, outfits_m5, latencias_m5 = [], [], []
for e in EVAL_SET_M5:
    im = Image.open(e["image_path"]) if e.get("image_path") else None
    res = shopper_m5.consultar(e["input"], imagen=im, registrar_log=True)
    respuestas_m5.append(res["respuesta"])
    outfits_m5.append(res["outfits"])
    latencias_m5.append(res["latencia_s"])

vram_m5_gb = round(torch.cuda.max_memory_allocated() / (1024**3), 2)

# Prueba de invarianza visual sobre la versión optimizada M5
img_base_path = "final/imagenes_test/ropa_01.png"
img_orig = Image.open(img_base_path).convert("RGB")
w, h = img_orig.size
mw, mh = int(w * 0.075), int(h * 0.075)
variantes = {
    "original": img_orig,
    "flip_horizontal": ImageOps.mirror(img_orig),
    "brillo_mas_20": ImageEnhance.Brightness(img_orig).enhance(1.20),
    "recorte_85pct": img_orig.crop((mw, mh, w - mw, h - mh)).resize((w, h)),
}
inv_res = {}
for k, im_v in variantes.items():
    r = shopper_m5.consultar("Quiero un outfit que combine con esta prenda para salir el fin de semana.", imagen=im_v, registrar_log=False)
    inv_res[k] = r

base_set = set(inv_res["original"]["outfits"])
df_inv_m5 = pd.DataFrame([
    {
        "variante": k,
        "es_moda_detectado": v["info_visual"]["es_moda"],
        "valvula_activada": v["valvula_activada"],
        "solapamiento_outfits_vs_orig": round(len(base_set & set(v["outfits"])) / max(1, len(base_set)), 2),
        "top_3_outfits": ", ".join(v["outfits"][:3]),
    }
    for k, v in inv_res.items()
])
df_inv_m5.to_csv("final/resultados/invarianza_m5.csv", index=False)
display(df_inv_m5)

# Liberar modelos de inferencia antes de cargar el juez independiente para evaluar D2 y D3
del shopper_m5
gc.collect()
torch.cuda.empty_cache()

# Evaluar D1, D2 y D3 con el mismo juez externo de M4
from google.colab import userdata
from huggingface_hub import login
try:
    login(token=userdata.get("HF_TOKEN"))
    JUDGE_ID = "google/gemma-2-2b-it"
except Exception:
    JUDGE_ID = "microsoft/Phi-3.5-mini-instruct"

_ID_RE = re.compile(r"[BC]_b\d+")
import csv
with open("final/corpus/corpus_look_es.csv", encoding="utf-8-sig") as f:
    _IDS_VALIDOS = {r["outfitID"] for r in csv.DictReader(f)}

d1_scores = []
for e, r, rec in zip(EVAL_SET_M5, respuestas_m5, outfits_m5):
    citados = set(_ID_RE.findall(r or ""))
    reales = {c for c in citados if c in _IDS_VALIDOS}
    if e["tipo"] == "fuera_de_corpus":
        d1_scores.append(1.0 if not citados else 0.0)
    elif not citados:
        d1_scores.append(0.0)
    else:
        d1_scores.append(sum(1 for c in reales if c in set(rec or [])) / len(citados))

RUBRICA_QA = (
    "Eres un juez experto evaluando a un personal shopper de moda basado en un catálogo. Te doy la "
    "PREGUNTA, la RESPUESTA ESPERADA y la RESPUESTA DEL SISTEMA. Califica la RESPUESTA DEL SISTEMA "
    "de 1 a 5:\n"
    "5 = recomienda outfits del catálogo coherentes, los cita y no inventa; o activa correctamente "
    "la válvula de escape cuando la pregunta o imagen está fuera del catálogo.\n"
    "4 = buena con un detalle flojo.\n3 = parcial (recupera algo pero se desvía o no cita).\n"
    "2 = pobre o poco fundamentada.\n1 = inventa prendas/datos, o NO activa la válvula cuando debía.\n"
    'Responde SOLO un JSON: {"puntaje": <1-5>, "motivo": "..."}.'
)

j_tok = AutoTokenizer.from_pretrained(JUDGE_ID)
j_mod = AutoModelForCausalLM.from_pretrained(JUDGE_ID, torch_dtype=torch.float16).to("cuda")
j_mod.eval()

d2_scores = []
with torch.no_grad():
    for e, r in zip(EVAL_SET_M5, respuestas_m5):
        pj = f"{RUBRICA_QA}\n\nPREGUNTA: {e['input']}\n\nRESPUESTA ESPERADA: {e['expected_response']}\n\nRESPUESTA DEL SISTEMA: {r}\n\nJSON:"
        inp = j_tok(j_tok.apply_chat_template([{"role": "user", "content": pj}], tokenize=False, add_generation_prompt=True), return_tensors="pt").to("cuda")
        out = j_mod.generate(**inp, max_new_tokens=120, do_sample=False, pad_token_id=j_tok.eos_token_id)
        raw = j_tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        try:
            p = int(parse_json_seguro(raw).get("puntaje"))
        except Exception:
            p = 3
        d2_scores.append(min(5, max(1, p)))

d3_scores = [
    (1 if VALVULA.lower() in (r or "").lower() else 0) if e["tipo"] == "fuera_de_corpus"
    else (1 if (p >= 4 and VALVULA.lower() not in (r or "").lower()) else 0)
    for e, r, p in zip(EVAL_SET_M5, respuestas_m5, d2_scores)
]

# Construir tabla comparativa oficial Antes (M4) vs Después (M5)
df_m4_prev = pd.read_csv("notebooks/M4/resultados/scorecard_m4.csv")
fila_m4 = df_m4_prev[df_m4_prev["sistema"].str.contains("M4 Completo")].iloc[0].to_dict()
fila_m4["sistema"] = "Antes — M4 Completo (FP16 sin optimizar)"

fila_m5 = {
    "sistema": "Después — M5 Optimizado (4-bit NF4 + HyDE estructurado)",
    "n_casos": len(EVAL_SET_M5),
    "D1_citacion": round(float(np.mean(d1_scores)), 3),
    "D2_juez": round(float(np.mean(d2_scores)), 3),
    "D3_cumplimiento": round(float(np.mean(d3_scores)), 3),
    "latencia_p50_s": round(float(np.percentile(latencias_m5, 50)), 3),
    "latencia_p95_s": round(float(np.percentile(latencias_m5, 95)), 3),
    "vram_max_gb": vram_m5_gb,
}

df_comp = pd.DataFrame([fila_m4, fila_m5])
df_comp.to_csv("final/resultados/comparacion_m4_vs_m5.csv", index=False)
display(df_comp)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

... (34 outputs left out)

model-00002-of-00002.safetensors:   0%|          | 0.00/241M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

,sistema,n_casos,D1_citacion,D2_juez,D3_cumplimiento,latencia_p50_s,latencia_p95_s,vram_max_gb
0,Antes — M4 Completo (FP16 sin optimizar),18,0.833,3.889,0.722,10.476,16.837,12.45
1,Después — M5 Optimizado (4-bit NF4 + HyDE estr...,18,0.750,3.611,0.611,11.375,16.319,3.22
